In [ ]:
from transformers import AutoTokenizer

# Load tokenizer of model that should be fine-tuned
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-6.7b-base")

In [ ]:
from spider_ent_data import get_spider_ent_data
spider_ent_data = get_spider_ent_data(tokenizer)

In [ ]:
spider_ent_data
print()

In [ ]:
from spider_data import get_spider_train, get_spider_val

spider_train_data = get_spider_train(tokenizer)
spider_val_data = get_spider_val(tokenizer)

In [5]:
import re
from transformers import AutoTokenizer
from spider_data import get_spider_val

tok = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-6.7b-base")
data = get_spider_val(tok, 3000)

miss, total = 0, 0
for item in data:
    cand = set()
    for prompt in item["input"]:
        # Regex-Lücken sichtbar machen:
        n_marker = prompt.count("«")
        found = re.findall(r"«\s+(\S+)\s+(\S+)\s*»", prompt)
        if len(found) != n_marker:
            print(f"Regex verliert {n_marker - len(found)} Kandidaten")
        cand |= {(t.lower(), c.lower()) for t, c in found}
    gold = {(t.lower(), c.lower()) for t, cols in item["gold_schema"].items() for c in (cols or [])}
    missing = gold - cand
    miss += len(missing); total += len(gold)
    if missing and miss < 30:
        print("Fehlt:", missing, "| Kandidaten-Beispiel:", sorted(cand)[:3])

print(f"\nStrukturell unerreichbar: {miss}/{total} = {miss/total:.2%}")

Fehlt: {('t1', 'stadium_id')} | Kandidaten-Beispiel: [('concert', 'concert_id'), ('concert', 'concert_name'), ('concert', 'stadium_id')]
Fehlt: {('t1', 'stadium_id')} | Kandidaten-Beispiel: [('concert', 'concert_id'), ('concert', 'concert_name'), ('concert', 'stadium_id')]
Fehlt: {('t1', 'stadium_id'), ('t1', 'year')} | Kandidaten-Beispiel: [('concert', 'concert_id'), ('concert', 'concert_name'), ('concert', 'stadium_id')]
Fehlt: {('t1', 'stadium_id'), ('t1', 'year')} | Kandidaten-Beispiel: [('concert', 'concert_id'), ('concert', 'concert_name'), ('concert', 'stadium_id')]
Fehlt: {('t1', 'stadium_id'), ('t1', 'year')} | Kandidaten-Beispiel: [('concert', 'concert_id'), ('concert', 'concert_name'), ('concert', 'stadium_id')]
Fehlt: {('t1', 'stadium_id'), ('t1', 'year')} | Kandidaten-Beispiel: [('concert', 'concert_id'), ('concert', 'concert_name'), ('concert', 'stadium_id')]
Fehlt: {('t1', 'concert_id')} | Kandidaten-Beispiel: [('concert', 'concert_id'), ('concert', 'concert_name'), ('co

In [2]:
pip install dotenv

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)
Note: you may need to restart the kernel to use updated packages.


In [3]:
import json
dev = json.load(open("data/spider/dev.json"))  # Pfad ggf. anpassen
bad = [ex for ex in dev if ex["db_id"] == "concert_singer" and " AS T1" in ex["query"].upper()]
ex = bad[0]
print(ex["question"])
print(ex["query"])

FileNotFoundError: [Errno 2] No such file or directory: 'data/spider/dev.json'

In [3]:
import re
from transformers import AutoTokenizer
from spider_data import get_spider_val, get_spider_train

tok = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-6.7b-base")
data = get_spider_train(tok, 3000)
for item in data:
    print(item['gold_schema'], item['sql'])

{} SELECT count(*) FROM head WHERE age  >  56
{} SELECT name ,  born_state ,  age FROM head ORDER BY age
{} SELECT creation ,  name ,  budget_in_billions FROM department
{} SELECT max(budget_in_billions) ,  min(budget_in_billions) FROM department
{} SELECT avg(num_employees) FROM department WHERE ranking BETWEEN 10 AND 15
{} SELECT name FROM head WHERE born_state != 'California'
{'t1': ['department_id', 'creation'], 'management': ['department_id', 'head_id'], 'head': ['born_state', 'head_id']} SELECT DISTINCT T1.creation FROM department AS T1 JOIN management AS T2 ON T1.department_id  =  T2.department_id JOIN head AS T3 ON T2.head_id  =  T3.head_id WHERE T3.born_state  =  'Alabama'
{} SELECT born_state FROM head GROUP BY born_state HAVING count(*)  >=  3
{} SELECT creation FROM department GROUP BY creation ORDER BY count(*) DESC LIMIT 1
{'t1': ['department_id', 'name', 'num_employees'], 'management': ['department_id', 'temporary_acting']} SELECT T1.name ,  T1.num_employees FROM departm

In [4]:
import re
from transformers import AutoTokenizer
from spider_data import get_spider_val, get_spider_train

tok = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-6.7b-base")
data = get_spider_train(tok, 3000)
item = data[0]
print(item['gold_schema'], item['sql'])

{} SELECT count(*) FROM head WHERE age  >  56
